# Building MCP Clients

Practice the client-side patterns that are easy to get wrong: routing mixed content blocks, building correctly-shaped `tool_result` messages, running the loop, and detecting termination. All exercises use mock LLM responses - no API key required.

***Summary***
1. [Setup - mock response objects](#setup)
2. [Classify a response](#classify)
3. [Build a `tool_result` message](#tool-result)
4. [Reconstruct message history from a trace](#reconstruct)
5. [Implement the process_query loop](#process-query)
6. [Generalization - safety rail against runaway loops](#generalization)


***
<a id='setup'></a>
## 1. Setup - mock response objects

The real Anthropic SDK returns objects like `response.content[i].type == 'text'`. We build small dataclass shims with the same shape so this notebook runs offline.


In [ ]:
from dataclasses import dataclass, field
from typing import List, Any


@dataclass
class MockText:
    text: str
    type: str = "text"


@dataclass
class MockToolUse:
    id: str
    name: str
    input: dict
    type: str = "tool_use"


@dataclass
class MockResponse:
    content: List[Any] = field(default_factory=list)

print("Mock classes ready")


***
<a id='classify'></a>
## 2. Classify a response

The client loop terminates when the response is text-only. Classification is the single most important decision the loop makes.


**Q1) Implement `classify_response(response) -> str` returning one of `"text_only"`, `"tool_only"`, or `"mixed"`.** Multiple text blocks is still `"text_only"`; multiple tool_use blocks is `"tool_only"`.

*Hint:* Build the set of block types with a [set comprehension](https://docs.python.org/3/tutorial/datastructures.html#sets) and compare it to known values.


In [ ]:
### YOUR CODE HERE ###
def classify_response(response) -> str:
    ...


In [ ]:
def test_classify_response():
    r1 = MockResponse(content=[MockText("Hello")])
    assert classify_response(r1) == "text_only"

    r2 = MockResponse(content=[MockToolUse(id="1", name="f", input={})])
    assert classify_response(r2) == "tool_only"

    r3 = MockResponse(content=[MockText("hmm"), MockToolUse(id="1", name="f", input={})])
    assert classify_response(r3) == "mixed"

    r4 = MockResponse(content=[MockText("one"), MockText("two")])
    assert classify_response(r4) == "text_only", "Multiple text blocks stay text-only"

    print("PASS: classify_response")

test_classify_response()


***
<a id='tool-result'></a>
## 3. Build a `tool_result` message

Given a `MockToolUse` block and the string result from executing that tool, produce the exact message dict that should be appended to `messages`.

Common mistakes:

- `role` is `"user"`, not `"assistant"` (even though *you* produced this message).
- `content` is a *list* containing one dict, not a bare dict.
- The inner field name is `tool_use_id`, not `id`, and it must match the block's `id` exactly.


**Q2) Implement `build_tool_result_message(tool_use_block, result) -> dict`.**

*Hint:* Copy the expected shape from Notebook 02, Part 4.1.


In [ ]:
### YOUR CODE HERE ###
def build_tool_result_message(tool_use_block, result: str) -> dict:
    ...


In [ ]:
def test_build_tool_result_message():
    block = MockToolUse(id="toolu_abc123", name="search_papers", input={"topic": "ml"})
    msg = build_tool_result_message(block, "paper1, paper2, paper3")

    assert msg["role"] == "user", f"role must be 'user', got {msg.get('role')!r}"
    assert isinstance(msg["content"], list), "content must be a list"
    assert len(msg["content"]) == 1

    inner = msg["content"][0]
    assert inner["type"] == "tool_result"
    assert inner["tool_use_id"] == "toolu_abc123"
    assert inner["content"] == "paper1, paper2, paper3"

    print("PASS: build_tool_result_message")

test_build_tool_result_message()


***
<a id='reconstruct'></a>
## 4. Reconstruct message history from a trace

Given a starting user query and a sequence of `(response, tool_result_string)` pairs, produce the final `messages` list. Ignore actual API calls - just build the correct list.

Rule per round: append the assistant response, then (if it had `tool_use` blocks) append the `tool_result` message.


**Q3) Implement `reconstruct_messages(initial_query, rounds) -> List[dict]`.** `rounds` is a list of `(MockResponse, str_or_None)` tuples. When the response is text-only, `tool_result` is `None` and no `tool_result` message is appended.

*Hint:* Start with `[{"role": "user", "content": initial_query}]`, then walk the rounds.


In [ ]:
### YOUR CODE HERE ###
def reconstruct_messages(initial_query: str, rounds: list) -> list:
    ...


In [ ]:
def test_reconstruct_messages():
    r1 = MockResponse(content=[MockToolUse(id="t1", name="search", input={"q": "ml"})])
    r2 = MockResponse(content=[MockText("Found 3 papers.")])

    messages = reconstruct_messages(
        initial_query="Find ML papers",
        rounds=[(r1, "p1, p2, p3"), (r2, None)],
    )

    assert len(messages) == 4, f"Expected 4 (user, asst, user tool_result, asst), got {len(messages)}"
    assert messages[0] == {"role": "user", "content": "Find ML papers"}
    assert messages[1]["role"] == "assistant"
    assert messages[2]["role"] == "user"
    assert messages[2]["content"][0]["type"] == "tool_result"
    assert messages[3]["role"] == "assistant"

    print("PASS: reconstruct_messages")

test_reconstruct_messages()


***
<a id='process-query'></a>
## 5. Implement the process_query loop

Build the complete client loop against a scripted "LLM". Instead of calling a real API, your loop pops the next response from a queue. The loop should:

1. Pop the first response.
2. For text blocks: record the text in an output list.
3. For tool_use blocks: call `tool_executor(name, args)` and (in a real system) send the result back - here, just pop the next response.
4. Stop when a response contains only text blocks (or when the queue empties).


**Q4) Implement `run_loop(initial_query, mock_responses, tool_executor) -> list`.** Return the list of user-visible text strings, in order.

*Hint:* Copy the shape of `handle_response` from Notebook 02, then wrap it in a `while True` that pops from `mock_responses`.


In [ ]:
### YOUR CODE HERE ###
def run_loop(initial_query: str, mock_responses: list, tool_executor) -> list:
    ...


In [ ]:
def test_run_loop():
    responses = [
        MockResponse(content=[MockToolUse(id="t1", name="search", input={"q": "ml"})]),
        MockResponse(content=[MockText("I found 3 papers on ML.")]),
    ]

    def executor(name, args):
        return "paper1, paper2, paper3"

    texts = run_loop("Find ML papers", responses, executor)
    assert texts == ["I found 3 papers on ML."], f"Got: {texts}"

    responses2 = [MockResponse(content=[MockText("Hi!")])]
    assert run_loop("Hello", responses2, executor) == ["Hi!"]

    responses3 = [
        MockResponse(content=[MockToolUse(id="t1", name="search", input={})]),
        MockResponse(content=[MockToolUse(id="t2", name="extract_info", input={})]),
        MockResponse(content=[MockText("Done.")]),
    ]
    assert run_loop("Q", responses3, executor) == ["Done."]

    print("PASS: run_loop")

test_run_loop()


***
<a id='generalization'></a>
## 6. Generalization - safety rail against runaway loops

A hostile or confused model may call the same tool with the same arguments forever. `run_loop` has no upper bound; a production loop needs one.

**Q5) Implement `run_loop_safe(initial_query, mock_responses, tool_executor, max_repeat=3)`.** If the same `(tool_name, tool_args)` pair is invoked more than `max_repeat` times, break out of the loop and return the collected texts plus an error message like `"Loop cut off - repeat detected on <tool>."`.

Design questions to think through as you write it (answer in the markdown cell after):

- How do you compare `tool_args` dicts? Direct equality is brittle if the model reorders keys.
- Should you count total calls, or only consecutive repeats?
- What should the final message tell the user?


In [ ]:
### YOUR CODE HERE ###
def run_loop_safe(initial_query, mock_responses, tool_executor, max_repeat=3):
    ...


*ANSWER HERE*

(How did you compare tool_args? Consecutive repeats or total repeats? What does the user see when the safety rail fires?)
